In [1]:
import sys
from tqdm import tqdm
import yaml
import os
from typing import List, Tuple, Dict, Generator
import json
import numpy as np
import joblib
import sys
import json
from tqdm import tqdm
from typing import Dict
import numpy as np
import torch
import os
from time import time
import gc

import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

[nltk_data] Error loading punkt_tab: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>
[nltk_data] Error loading wordnet: <urlopen error [Errno -2] Name or
[nltk_data]     service not known>


In [2]:
from src.kg_model import KnowledgeGraphModel
from src.pipelines.qa import QAPipelineConfig, QAPipeline


from src.pipelines.qa.knowledge_retriever import AStarGraphSearchConfig, AStarMetricsConfig, BFSSearchConfig, MixturedGraphSearchConfig
from src.pipelines.qa.knowledge_retriever.TripletsFilter import TripletsFilterConfig
from src.pipelines.qa import QueryLLMParserConfig, KnowledgeComparatorConfig, KnowledgeRetrieverConfig, QALLMGeneratorConfig


from src.utils import NodeType, Logger

/home/dzigen/Desktop/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Create configs

In [3]:
# retrieve
RETRIVER_CONFIG_DUMP = "retriever_config" 

retriever_config = BFSSearchConfig(
    strict_filter=True,
    hyper_episodic_num=15,
    chain_triplets_num=25,
    other_triplets_num=6
)

joblib.dump(retriever_config, RETRIVER_CONFIG_DUMP)


# filter config
FILTER_CONFIG_DUMP = "filter_config"

filter_config = TripletsFilterConfig(
    max_k=50
)

joblib.dump(filter_config, FILTER_CONFIG_DUMP)

['filter_config']

## Loading hyperparameters

In [3]:
# Read YAML file
with open("params.yaml", 'r') as stream:
    HYPER_PARAMS = yaml.safe_load(stream)

BASE_PATH = "../../data/knowledge_graphs/"
DATASET_PATH = BASE_PATH + f"{HYPER_PARAMS['dataset_name']}/"
KG_PATH = DATASET_PATH + f"{HYPER_PARAMS['kg_name']}/"

GRAPH_DRIVER_CONFIG_PATH = KG_PATH + "graph_config"
EMBEDDINGS_DRIVER_CONFIG_PATH = KG_PATH + "embeddings_config"

EXPERIMENT_DIR = f"{HYPER_PARAMS['dataset_name']}/exp_logs/{HYPER_PARAMS['experiment_name']}"
GENERATED_ANSWERS_DIR = f'{EXPERIMENT_DIR}/answer_packs'
METRICS_DIR = f'{EXPERIMENT_DIR}/metric_packs'

TMP_GENERATED_ANSWERS_DIR = f'{EXPERIMENT_DIR}/tmp_answer_packs'

QA_ELAPSED_TIME = f'{EXPERIMENT_DIR}/elapsed_time.json'

HYPERPARAMS_SAVE_PATH = f'{EXPERIMENT_DIR}/hyperparams.json'
QA_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/qa_config'
RETRIEVER_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/retirver_config'
FILTER_CONFIG_SAVE_PATH = f'{EXPERIMENT_DIR}/filter_config'

In [4]:
# инициализируем граф знаний

graph_config = joblib.load(GRAPH_DRIVER_CONFIG_PATH)
embed_config = joblib.load(EMBEDDINGS_DRIVER_CONFIG_PATH)

# !!! IMPORTANT !!!
graph_config.driver_config.db_config.need_to_clear = False
embed_config.nodesdb_driver_config.db_config.need_to_clear = False
embed_config.tripletsdb_driver_config.db_config.need_to_clear = False
# !!! IMPORTANT !!!

# fixing paths to vector dbs
embed_config.embedder_config.model_name_or_path = '/'.join(embed_config.embedder_config.model_name_or_path.split("/")[1:])
embed_config.nodesdb_driver_config.db_config.path = '/'.join(embed_config.nodesdb_driver_config.db_config.path.split("/")[1:])
embed_config.tripletsdb_driver_config.db_config.path = '/'.join(embed_config.tripletsdb_driver_config.db_config.path.split("/")[1:])

kg_model = KnowledgeGraphModel(
    graph_config=graph_config,
    embeddings_config=embed_config)

print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


49597
44328
{'triplets': 182838, 'nodes': 49597}


In [5]:
# задаём конфигурацию qa-пайплайна

retriever_config = joblib.load(HYPER_PARAMS['knowledge_retriever']['retriever_config_path'])
filter_config = joblib.load(HYPER_PARAMS['knowledge_retriever']['filter_config_path'])

qa_config = QAPipelineConfig(
    query_parser_config=QueryLLMParserConfig(lang=HYPER_PARAMS['language']),

    knowledge_comparator_config=KnowledgeComparatorConfig(),
    
    knowledge_retriever_config=KnowledgeRetrieverConfig(
        retriever_method=HYPER_PARAMS['knowledge_retriever']['retriever_method'], retriever_config=retriever_config,
        filter_method=HYPER_PARAMS['knowledge_retriever']['filter_method'], filter_config=filter_config),
    
    answer_generator_config=QALLMGeneratorConfig(lang=HYPER_PARAMS['language']))

In [6]:
qa_pipeline = QAPipeline(kg_model, qa_config)

## Structure init

In [7]:
if not os.path.exists(HYPER_PARAMS['dataset_name']):
    raise ValueError("Директории не существует")

if os.path.exists(EXPERIMENT_DIR):
    raise ValueError("Директория существует")

if os.path.exists(GENERATED_ANSWERS_DIR):
    raise ValueError("Директория существует")

if os.path.exists(METRICS_DIR):
    raise ValueError("Директория существует")

# создать каталог
os.mkdir(EXPERIMENT_DIR)
# создать каталог для ответов
os.mkdir(GENERATED_ANSWERS_DIR)
# создать каталог для метрик
os.mkdir(METRICS_DIR)

os.mkdir(TMP_GENERATED_ANSWERS_DIR)

# сохранить параметры
with open(HYPERPARAMS_SAVE_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(HYPER_PARAMS, indent=1, ensure_ascii=False))

# сохранить конфиги
joblib.dump(qa_config, QA_CONFIG_SAVE_PATH)
joblib.dump(retriever_config, RETRIEVER_CONFIG_SAVE_PATH)
joblib.dump(filter_config, FILTER_CONFIG_SAVE_PATH)

['diaasqa/exp_logs/bfs_with_gigachat_full/filter_config']

## Loading questions

In [8]:
def diaasqa_loading(dir_path: str):
    pack_files = os.listdir(dir_path)
    packs = []

    for pack_f in pack_files:
        with open(f"{dir_path}/{pack_f}", 'r', encoding='utf-8') as fd:
            data = json.loads(fd.read())

        pack_name = '.'.join(pack_f.split('.')[:-1])
        questions = list(map(lambda item: item['question'], data)) 
        answers = list(map(lambda item: item['answer'], data))

        packs.append((pack_name, questions, answers))

    return packs

In [9]:
DATASET_LOADERS = {
    'diaasqa': diaasqa_loading
}

## Starting qa process

In [10]:
question_packs = DATASET_LOADERS[HYPER_PARAMS['dataset_name']](HYPER_PARAMS['eval_dataset_path'])

In [19]:
for pack_name, questions, _ in question_packs[11:]:
    
    pack_tmp_dir = f"{TMP_GENERATED_ANSWERS_DIR}/{pack_name}"
    if not os.path.exists(pack_tmp_dir):
        os.mkdir(pack_tmp_dir)

    process = tqdm(range(len(questions)))
    for i in process:
        process.set_postfix_str(pack_name)

        s_time = time()
        answer, info = qa_pipeline.answer(questions[i])
        e_time = time()

        answer_dump_file = f"{pack_tmp_dir}/answer_{i}"
        joblib.dump({'answer': answer, 'info': info, 'elapsed_time': e_time - s_time}, answer_dump_file)

100%|██████████| 200/200 [10:10<00:00,  3.05s/it, device_sentiment]


In [22]:
# accumulate generate answers
elapsed_times = {}

for pack_name, _, _ in question_packs:

    pack_tmp_dir = f"{TMP_GENERATED_ANSWERS_DIR}/{pack_name}"

    if not os.path.exists(pack_tmp_dir):
        print("Папки с ответами не сущестует: ", pack_name)
        continue

    tmp_answer_dumps = os.listdir(pack_tmp_dir)

    accum_answers = dict()
    elapsed_times[pack_name] = {'per_question': []}
    for tmp_dump in tqdm(tmp_answer_dumps):
        answer_info = joblib.load(f"{pack_tmp_dir}/{tmp_dump}")
        answer_num = int(tmp_dump.split("_")[1])
        accum_answers[answer_num] = answer_info['answer']
        elapsed_times[pack_name]['per_question'].append(answer_info['elapsed_time'])

    elapsed_times[pack_name]['sum'] = sum(elapsed_times[pack_name]['per_question'])
    elapsed_times[pack_name]['mean'] = np.mean(elapsed_times[pack_name]['per_question'])
    elapsed_times[pack_name]['median'] = np.median(elapsed_times[pack_name]['per_question'])
    
    answers_pack_path = f"{GENERATED_ANSWERS_DIR}/{pack_name}.json"
    with open(answers_pack_path, 'w', encoding='utf-8') as fd:
        fd.write(json.dumps(accum_answers, indent=1, ensure_ascii=False))

with open(QA_ELAPSED_TIME, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(elapsed_times, indent=1, ensure_ascii=False))

100%|██████████| 200/200 [00:00<00:00, 14664.89it/s]


## Measuring answers quality

In [ ]:
def loading_generated_pack(base_dir: str, pack_name) -> Dict[int,str]:
    with open(f"{base_dir}/{pack_name}", 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())
    return data

def round5(number: float) -> float:
    return round(number, 5)

def save_json(data: Dict[str, object], save_path: str):
    dump = json.dumps(data, ensure_ascii=False, indent=1)
    with open(f"{save_path}.json", 'w', encoding='utf-8') as fd:
        fd.write(dump)

In [ ]:
BERTSCORE_MODEL_PATH = "google/electra-base-discriminator"
METRICS = ReaderMetrics(base_dir="../..", bs_model_path=BERTSCORE_MODEL_PATH)

In [ ]:
# загружаем датасте 
question_packs = DATASET_LOADERS(HYPER_PARAMS['diaasqa'])(HYPER_PARAMS['eval_dataset_path'])

for pack_name, _, all_target_answers in tqdm(question_packs):

    torch.cuda.empty_cache()
    gc.collect()

    generated_pack = loading_generated_pack(GENERATED_ANSWERS_DIR, pack_name)
    
    filtered_target_answers = []
    generated_answers = []
    for i, answer in generated_pack.items():
        generated_answers.append(answer)
        filtered_target_answers.append(all_target_answers[i])

    b1_scores = METRICS.bleu1(generated_answers, filtered_target_answers)
    b2_scores  = METRICS.bleu2(generated_answers, filtered_target_answers)
    rl_scores = METRICS.rougel(generated_answers, filtered_target_answers)
    m_scores = METRICS.meteor(generated_answers, filtered_target_answers)
    em_scores = METRICS.exact_match(generated_answers, filtered_target_answers)
    bs_scores = METRICS.bertscore(generated_answers, filtered_target_answers)

    scores = {
        'BLEU1': b1_scores,
        'BLEU2': b2_scores,
        'METEOR': m_scores,
        'RougeL': rl_scores,
        'ExactMatch': em_scores,
        'BertScore': bs_scores,
        'BertScore_model': BERTSCORE_MODEL_PATH
    }

    # сохраняем скоры по папку
    save_json(scores, f"{METRICS_DIR}/{pack_name}")